# MLP-TUNE-001 — подбор гиперпараметров Optuna

Этот notebook не создаёт 30 официальных `MLP-xxx`. Он проводит screening вокруг
принятой **MLP-024**, сохраняет каждую trial в SQLite и показывает лучшие наборы.

Порядок: **фиксированные folds → trials → таблица результатов → проверка лучших
конфигураций → отдельный официальный MLP-эксперимент**.

`test.csv` в подборе не используется. Запускайте notebook в одном kernel и после
работы завершайте его через **Kernel → Shut Down Kernel**.


## 1. Ограничиваем вычислительные потоки

Эта ячейка должна выполняться первой. `n_jobs=1` означает, что Optuna не создаёт
параллельные trials, а ограничения BLAS не позволяют одному kernel резервировать
гигабайты памяти под множество потоков.


In [1]:
import os

for variable in (
    "OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "OMP_THREAD_LIMIT",
    "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS",
):
    os.environ[variable] = "1"


## 2. Импорты и источник данных

Да, здесь впервые появляется `import optuna`. Но сначала пакет должен быть частью
окружения (`requirements.txt` и `environment.yml` уже обновлены).


In [2]:
from pathlib import Path
import sys

import optuna
import pandas as pd
import torch
from IPython.display import display

torch.set_num_threads(1)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (path / "README.md").is_file() and (path / "src").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.mlp_experiment as mlp_tools
import ml_project.mlp_tuning as tuning

PARENT_MODULE = "ml_project.mlp_experiments.mlp_024_dpoutmedium"
parent = mlp_tools.load_mlp_experiment(PARENT_MODULE)

catalog = DataCatalog(PROJECT_ROOT, project_config.RAW_DIR, project_config.DATASETS)
raw_train = catalog.load(project_config.TRAIN_DATASET)
prepared = parent.prepare_features(raw_train.copy(deep=True))
mlp_tools.validate_feature_data(
    prepared, raw_train, target=project_config.TARGET, key=project_config.KEY
)

build_fold_transformer = getattr(parent, "build_fold_transformer", None)
cv_splits = tuning.make_stratified_splits(
    prepared,
    target=project_config.TARGET,
    n_splits=3,
    random_state=2026,
)

print("Project:", PROJECT_ROOT)
print("Parent:", parent.EXPERIMENT.experiment_id, parent.EXPERIMENT.title)
print("Rows:", len(prepared.frame))
print("Screening folds:", len(cv_splits))


C:\ML_Progs\Runtimes\Miniforge3\envs\titanik-ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: D:\Projects\Titanik-ML-project
Parent: MLP-024 dpoutmedium
Rows: 891
Screening folds: 3


## 3. Пространство поиска

Optuna будет выбирать batch size, learning rate, weight decay, Dropout, глубину и
общую ширину скрытых слоёв. ReLU, положение Dropout, признаки и preprocessing
зафиксированы результатами ручных экспериментов.

Первая trial принудительно повторяет параметры MLP-024 и становится baseline на
новых screening folds. Её accuracy не обязана равняться официальным `0.8384`:
здесь используются три folds с другим seed. Все последующие trials сравниваются
именно с этой контрольной trial на тех же строках.


In [3]:
STUDY_NAME = "titanic_mlp_tuning_001"
STUDY_DIR = PROJECT_ROOT / "artifacts/optuna"
STUDY_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_PATH = STUDY_DIR / f"{STUDY_NAME}.db"
STORAGE = f"sqlite:///{DATABASE_PATH.as_posix()}"

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

if not study.trials:
    study.enqueue_trial(tuning.BASELINE_PARAMETERS)

objective = tuning.make_objective(
    prepared,
    target=project_config.TARGET,
    key=project_config.KEY,
    base_spec=parent.EXPERIMENT,
    cv_splits=cv_splits,
    build_fold_transformer=build_fold_transformer,
)

print("Storage:", DATABASE_PATH.relative_to(PROJECT_ROOT))
print("Existing trials:", len(study.trials))
display(pd.Series(tuning.BASELINE_PARAMETERS, name="MLP-024").to_frame())


[I 2026-10-05 13:43:17,871] Using an existing study with `study_name='titanic_mlp_tuning_001'` instead of creating a new one.


Storage: artifacts\optuna\titanic_mlp_tuning_001.db
Existing trials: 1


,MLP-024
batch_size,32.000
learning_rate,0.001
weight_decay,0.000
dropout,0.200
n_layers,3.000
hidden_dim,16.000


## 4. Запуск trials

Для первого знакомства начните с 10 trials. Повторный запуск продолжит ту же
SQLite study. Когда всё понятно, измените `N_TRIALS` на 20–30.


In [8]:
N_TRIALS = 100

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    gc_after_trial=True,
    show_progress_bar=True,
)


Best trial: 19. Best value: 0.82716:   1%|          | 1/100 [00:07<11:50,  7.17s/it]

[I 2026-10-05 14:00:24,192] Trial 40 finished with value: 0.8170594837261503 and parameters: {'batch_size': 32, 'learning_rate': 0.0010963572386520648, 'weight_decay': 1e-06, 'dropout': 0.3, 'n_layers': 4, 'hidden_dim': 8}. Best is trial 19 with value: 0.8271604938271606.


Best trial: 19. Best value: 0.82716:   2%|▏         | 2/100 [00:14<11:44,  7.19s/it]

[I 2026-10-05 14:00:31,413] Trial 41 finished with value: 0.819304152637486 and parameters: {'batch_size': 32, 'learning_rate': 0.0015221298680985055, 'weight_decay': 1e-06, 'dropout': 0.3, 'n_layers': 4, 'hidden_dim': 8}. Best is trial 19 with value: 0.8271604938271606.


Best trial: 19. Best value: 0.82716:   3%|▎         | 3/100 [00:18<09:02,  5.60s/it]

...

[I 2026-10-05 14:08:26,975] Trial 137 finished with value: 0.8204264870931537 and parameters: {'batch_size': 64, 'learning_rate': 0.001299081815862803, 'weight_decay': 1e-06, 'dropout': 0.25, 'n_layers': 4, 'hidden_dim': 32}. Best is trial 86 with value: 0.8327721661054994.


Best trial: 86. Best value: 0.832772:  99%|█████████▉| 99/100 [08:11<00:03,  3.53s/it]

[I 2026-10-05 14:08:28,982] Trial 138 finished with value: 0.8249158249158249 and parameters: {'batch_size': 64, 'learning_rate': 0.0020647919144194254, 'weight_decay': 1e-06, 'dropout': 0.25, 'n_layers': 4, 'hidden_dim': 32}. Best is trial 86 with value: 0.8327721661054994.


Best trial: 86. Best value: 0.832772: 100%|██████████| 100/100 [08:17<00:00,  4.98s/it]

[I 2026-10-05 14:08:34,553] Trial 139 finished with value: 0.8181818181818182 and parameters: {'batch_size': 64, 'learning_rate': 0.0015940053829212974, 'weight_decay': 0.001, 'dropout': 0.1, 'n_layers': 4, 'hidden_dim': 8}. Best is trial 86 with value: 0.8327721661054994.


## 5. Результаты

`value` — средняя accuracy на трёх одинаковых screening folds. Эта таблица нужна
для отбора кандидатов, а не для объявления нового чемпиона.


In [9]:
trials = study.trials_dataframe()
trials.to_csv(STUDY_DIR / f"{STUDY_NAME}_trials.csv", index=False)

columns = [
    "number", "value", "state",
    "params_batch_size", "params_learning_rate", "params_weight_decay",
    "params_dropout", "params_n_layers", "params_hidden_dim",
    "user_attrs_accuracy_std", "user_attrs_log_loss_mean",
]
available = [column for column in columns if column in trials]
display(
    trials.loc[trials["state"].eq("COMPLETE"), available]
    .sort_values("value", ascending=False)
    .head(10)
)

print("Best screening accuracy:", study.best_value)
print("Best parameters:", study.best_params)


,number,value,state,params_batch_size,params_learning_rate,params_weight_decay,params_dropout,params_n_layers,params_hidden_dim,user_attrs_accuracy_std,user_attrs_log_loss_mean
86,86,0.832772,COMPLETE,32,0.001750,1.000000e-04,0.30,4,8,0.023888,0.454552
106,106,0.829405,COMPLETE,64,0.001720,1.000000e-06,0.10,4,8,0.030551,0.442854
69,69,0.829405,COMPLETE,32,0.001955,1.000000e-06,0.10,4,8,0.018544,0.443216
103,103,0.829405,COMPLETE,64,0.001853,1.000000e-06,0.10,4,8,0.023888,0.438748
111,111,0.829405,COMPLETE,64,0.001902,1.000000e-06,0.15,4,8,0.031824,0.435031
58,58,0.827160,COMPLETE,32,0.002316,1.000000e-06,0.25,4,8,0.022419,0.467705
129,129,0.827160,COMPLETE,64,0.001801,1.000000e-06,0.25,4,32,0.028036,0.428937
74,74,0.827160,COMPLETE,32,0.001527,1.000000e-07,0.10,4,8,0.023649,0.445297
19,19,0.827160,COMPLETE,32,0.001468,1.000000e-07,0.15,4,8,0.019146,0.459858
64,64,0.827160,COMPLETE,32,0.002934,1.000000e-07,0.10,4,8,0.024819,0.437724


Best screening accuracy: 0.8327721661054994
Best parameters: {'batch_size': 32, 'learning_rate': 0.0017503534730658975, 'weight_decay': 0.0001, 'dropout': 0.3, 'n_layers': 4, 'hidden_dim': 8}


## 6. Что делать с победителем

1. Не помечать trial как `adopt`.
2. Взять 3–5 лучших конфигураций и проверить их на официальных пяти folds и
   нескольких seed.
3. Создать `new-MLPexperiment.cmd` на основе MLP-024.
4. Перенести только выбранные параметры и архитектуру.
5. Выполнить обычный notebook, получить paired-сравнение и только затем принять
   или отклонить кандидата.

SQLite и CSV находятся в `artifacts/optuna/`; повторный запуск продолжает study.
